# Gaitkeeper v3: All 3 Attack Methods + Multi-Objective Loss
**Methods:** FGSM → PGD → EoT-PGD  
**Loss:** Uncertainty + IoU + Confidence + Edge  

> Set Runtime → T4 GPU before running all cells.

In [ ]:
!pip install ultralytics opencv-python-headless matplotlib scipy -q

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
import torchvision.transforms.functional as TF
import random
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Patch placement on person box
PATCH_SCALE  = 0.4
SHIRT_TOP    = 0.15
SHIRT_BOT    = 0.55

# Perturbation budget
EPSILON      = 0.1

# PGD
PGD_STEPS    = 40
PGD_ALPHA    = (EPSILON / PGD_STEPS) * 2.5

# EoT-PGD
EOT_STEPS    = 40
EOT_ALPHA    = (EPSILON / EOT_STEPS) * 2.5
EOT_N        = 10
LR           = 0.01

# Multi-objective loss weights
W_UNCERTAINTY = 0.4
W_IOU         = 0.3
W_CONFIDENCE  = 0.2
W_EDGE        = 0.1

# EoT transform ranges
ROT_RANGE        = (-20, 20)
SCALE_RANGE      = (0.85, 1.15)
BRIGHTNESS_RANGE = (0.7, 1.3)
CONTRAST_RANGE   = (0.8, 1.2)
NOISE_STD        = 0.05
BLUR_MAX         = 3
PERSPECTIVE_PROB = 0.5

YOLO_MODEL   = 'yolov8n-seg'
PERSON_CLASS = 0

print('Config loaded.')

In [ ]:
model = YOLO(YOLO_MODEL)
model.to(DEVICE)
torch_model = model.model
torch_model.eval()
print(f'Loaded {YOLO_MODEL}')

In [ ]:
import urllib.request

# Option A: upload from your machine
# from google.colab import files
# uploaded = files.upload()
# IMAGE_PATH = list(uploaded.keys())[0]

# Option B: use a local file already in Colab
# IMAGE_PATH = '/content/your_frame.jpg'

# Option C: placeholder (has people in it — works for initial testing)
urllib.request.urlretrieve('https://ultralytics.com/images/bus.jpg', '/content/test.jpg')
IMAGE_PATH = '/content/test.jpg'

orig_bgr = cv2.imread(IMAGE_PATH)
orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(8,6))
plt.imshow(orig_rgb)
plt.title('Input Image')
plt.axis('off')
plt.show()
print(f'Image shape: {orig_rgb.shape}')

In [ ]:
def run_inference(img_rgb, title='YOLOv8-seg'):
    results = model(img_rgb, verbose=False)
    result  = results[0]
    ann     = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)

    confs = []
    if result.boxes is not None:
        for i, c in enumerate(result.boxes.cls):
            if int(c) == PERSON_CLASS:
                confs.append(float(result.boxes.conf[i]))

    avg_conf  = float(np.mean(confs)) if confs else 0.0
    has_masks = result.masks is not None

    plt.figure(figsize=(10,7))
    plt.imshow(ann)
    plt.title(f'{title} | persons={len(confs)} conf={avg_conf:.3f} {"masks OK" if has_masks else "NO MASKS"}')
    plt.axis('off')
    plt.show()

    print(f'[{title}] persons={len(confs)}, avg_conf={avg_conf:.4f}, masks={has_masks}')
    return results, avg_conf

baseline_results, baseline_conf = run_inference(orig_rgb, 'Baseline (Clean)')
print(f'\nTarget to beat: {baseline_conf:.4f}')

In [ ]:
def img_to_tensor(img_rgb):
    t = torch.from_numpy(img_rgb).float() / 255.0
    return t.permute(2, 0, 1).unsqueeze(0).to(DEVICE)

def tensor_to_img(t):
    t = t.squeeze(0).permute(1, 2, 0).detach().cpu()
    return (t.clamp(0, 1) * 255).byte().numpy()

def get_shirt_box(results, img_h, img_w):
    result = results[0]
    if result.boxes is None:
        return None
    best_box, best_area = None, 0
    for i, cls in enumerate(result.boxes.cls):
        if int(cls) == PERSON_CLASS:
            box  = result.boxes.xyxy[i].cpu().numpy()
            area = (box[2]-box[0]) * (box[3]-box[1])
            if area > best_area:
                best_area, best_box = area, box
    if best_box is None:
        return None
    bx1, by1, bx2, by2 = best_box
    bw, bh = bx2-bx1, by2-by1
    cx = (bx1+bx2)/2
    x1 = max(0,       int(cx - bw*PATCH_SCALE/2))
    x2 = min(img_w-1, int(cx + bw*PATCH_SCALE/2))
    y1 = max(0,       int(by1 + bh*SHIRT_TOP))
    y2 = min(img_h-1, int(by1 + bh*SHIRT_BOT))
    return (x1, y1, x2, y2)

def apply_patch(img_tensor, patch, box):
    x1, y1, x2, y2 = box
    ph, pw = y2-y1, x2-x1
    if ph <= 0 or pw <= 0:
        return img_tensor
    resized = F.interpolate(patch, size=(ph, pw), mode='bilinear', align_corners=False)
    out = img_tensor.clone()
    out[:, :, y1:y2, x1:x2] = resized
    return out

def get_raw_output(img_tensor):
    resized = F.interpolate(img_tensor, size=(640, 640), mode='bilinear', align_corners=False)
    return torch_model(resized)

print('Helpers loaded.')

In [ ]:
img_h, img_w = orig_rgb.shape[:2]
shirt_box = get_shirt_box(baseline_results, img_h, img_w)

if shirt_box is None:
    print('ERROR: No person detected. Try a different image.')
else:
    x1, y1, x2, y2 = shirt_box
    patch_h, patch_w = y2-y1, x2-x1
    print(f'Shirt region: x=[{x1},{x2}], y=[{y1},{y2}], size={patch_w}x{patch_h}px')

    vis = orig_rgb.copy()
    cv2.rectangle(vis, (x1,y1), (x2,y2), (255,0,0), 3)
    cv2.putText(vis, 'patch here', (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,0,0), 2)
    plt.figure(figsize=(8,6))
    plt.imshow(vis)
    plt.title('Detected Shirt Region')
    plt.axis('off')
    plt.show()

    # Init patch as random noise in [0.4, 0.6] (neutral gray, room to move both directions)
    patch_init = (torch.rand(1, 3, patch_h, patch_w, device=DEVICE) * 0.2 + 0.4).clamp(0, 1)
    print(f'Patch initialized. Shape: {list(patch_init.shape)}')

In [ ]:
def apply_motion_blur(tensor, max_kernel=3):
    k = random.choice([1, 3]) if max_kernel >= 3 else 1
    if k == 1:
        return tensor
    kernel = torch.zeros(k, k, device=tensor.device)
    if random.random() > 0.5:
        kernel[k//2, :] = 1.0 / k
    else:
        kernel[:, k//2] = 1.0 / k
    kernel = kernel.view(1, 1, k, k).expand(3, 1, k, k)
    return F.conv2d(tensor, kernel, padding=k//2, groups=3)

def apply_perspective_warp(tensor):
    _, _, H, W = tensor.shape
    d = 0.05
    src = np.float32([[0,0],[W,0],[W,H],[0,H]])
    dst = np.float32([
        [random.uniform(0, d*W),     random.uniform(0, d*H)],
        [W-random.uniform(0, d*W),   random.uniform(0, d*H)],
        [W-random.uniform(0, d*W),   H-random.uniform(0, d*H)],
        [random.uniform(0, d*W),     H-random.uniform(0, d*H)],
    ])
    M      = cv2.getPerspectiveTransform(src, dst)
    img_np = tensor_to_img(tensor)
    warped = cv2.warpPerspective(img_np, M, (W, H))
    return img_to_tensor(warped)

def random_eot_transform(img_tensor):
    t = img_tensor.clone()
    # Brightness
    t = t * random.uniform(*BRIGHTNESS_RANGE)
    # Contrast
    mean = t.mean(dim=[2,3], keepdim=True)
    t = (t - mean) * random.uniform(*CONTRAST_RANGE) + mean
    # Rotation
    t = TF.rotate(t.squeeze(0), random.uniform(*ROT_RANGE)).unsqueeze(0)
    # Scale
    scale = random.uniform(*SCALE_RANGE)
    _, _, H, W = t.shape
    new_h, new_w = int(H*scale), int(W*scale)
    t_s = F.interpolate(t, size=(new_h, new_w), mode='bilinear', align_corners=False)
    if scale < 1.0:
        ph, pw = (H-new_h)//2, (W-new_w)//2
        t = F.pad(t_s, (pw, W-new_w-pw, ph, H-new_h-ph))
    else:
        sh, sw = (new_h-H)//2, (new_w-W)//2
        t = t_s[:, :, sh:sh+H, sw:sw+W]
    # Perspective warp
    if random.random() < PERSPECTIVE_PROB:
        t = apply_perspective_warp(t)
    # Motion blur
    t = apply_motion_blur(t, BLUR_MAX)
    # Noise
    t = t + torch.randn_like(t) * NOISE_STD
    return t.clamp(0, 1)

print('EoT transforms loaded.')

In [ ]:
def extract_preds(raw_output):
    """
    Handle ALL known YOLOv8 raw output formats:
    dict, tuple, list of tensors, bare tensor.
    Returns: class_logits [1,80,N], person_logits [1,N], preds [1,C,N]
    """
    # Unwrap dict
    if isinstance(raw_output, dict):
        for key in ['one2one', 'one2many', 'preds', 'output']:
            if key in raw_output:
                raw_output = raw_output[key]
                break
        else:
            for v in raw_output.values():
                if isinstance(v, (torch.Tensor, list, tuple)):
                    raw_output = v
                    break

    # Unwrap outer tuple/list
    if isinstance(raw_output, (tuple, list)):
        preds = raw_output[0]
    else:
        preds = raw_output

    # preds might be a list of per-scale tensors
    if isinstance(preds, (tuple, list)):
        flat = []
        for p in preds:
            if isinstance(p, torch.Tensor):
                flat.append(p.flatten(2) if p.dim() == 4 else p)
        if not flat:
            raise ValueError(f'No tensors found in preds: {[type(x) for x in preds]}')
        preds = torch.cat(flat, dim=2)

    # Unwrap nested dict
    if isinstance(preds, dict):
        for key in ['one2one', 'one2many', 'preds', 'output']:
            if key in preds:
                preds = preds[key]
                break
        else:
            preds = next(v for v in preds.values() if isinstance(v, torch.Tensor))

    if not isinstance(preds, torch.Tensor):
        raise ValueError(f'Could not extract tensor. Got: {type(preds)}')

    # Flatten 4D
    if preds.dim() == 4:
        preds = preds.flatten(2)

    # Transpose if [1, N, C] instead of [1, C, N]
    if preds.dim() == 3 and preds.shape[1] > preds.shape[2]:
        preds = preds.permute(0, 2, 1)

    class_logits  = preds[:, 4:84, :]
    person_logits = preds[:, 4, :]
    return class_logits, person_logits, preds


def entropy_loss(raw_output):
    class_logits, _, _ = extract_preds(raw_output)
    probs   = torch.softmax(class_logits, dim=1)
    entropy = -(probs * (probs + 1e-8).log()).sum(dim=1)
    return -entropy.mean()

def confidence_loss(raw_output):
    _, person_logits, _ = extract_preds(raw_output)
    return torch.sigmoid(person_logits).mean()

def iou_loss(raw_output, shirt_box, img_shape):
    _, person_logits, preds = extract_preds(raw_output)
    H, W = img_shape
    x1, y1, x2, y2 = shirt_box
    target = torch.zeros(1, 1, H, W, device=preds.device)
    target[:, :, y1:y2, x1:x2] = 1.0
    best_idx = torch.sigmoid(person_logits).argmax(dim=1)[0]
    cx = preds[0, 0, best_idx]
    cy = preds[0, 1, best_idx]
    bw = preds[0, 2, best_idx]
    bh = preds[0, 3, best_idx]
    scale_x, scale_y = W/640.0, H/640.0
    px1 = int(((cx-bw/2)*640*scale_x).clamp(0,W-1).item())
    py1 = int(((cy-bh/2)*640*scale_y).clamp(0,H-1).item())
    px2 = int(((cx+bw/2)*640*scale_x).clamp(0,W-1).item())
    py2 = int(((cy+bh/2)*640*scale_y).clamp(0,H-1).item())
    pred_mask = torch.zeros(1, 1, H, W, device=preds.device)
    if px2 > px1 and py2 > py1:
        pred_mask[:, :, py1:py2, px1:px2] = 1.0
    intersection = (pred_mask * target).sum()
    union        = pred_mask.sum() + target.sum() - intersection
    return intersection / (union + 1e-6)

def edge_loss(raw_output):
    _, person_logits, _ = extract_preds(raw_output)
    return -torch.sigmoid(person_logits).var()

def combined_loss(raw_output, shirt_box, img_shape):
    L_unc  = entropy_loss(raw_output)
    L_iou  = iou_loss(raw_output, shirt_box, img_shape)
    L_conf = confidence_loss(raw_output)
    L_edge = edge_loss(raw_output)
    total  = (W_UNCERTAINTY*L_unc + W_IOU*L_iou + W_CONFIDENCE*L_conf + W_EDGE*L_edge)
    return total, {
        'uncertainty': L_unc.item(),
        'iou':         L_iou.item(),
        'confidence':  L_conf.item(),
        'edge':        L_edge.item(),
        'total':       total.item()
    }

# Sanity check
print('Running format diagnostic...')
_t = img_to_tensor(orig_rgb)
_r = F.interpolate(_t, size=(640,640), mode='bilinear', align_corners=False)
torch_model.train()
with torch.no_grad():
    _out = torch_model(_r)
torch_model.eval()

def _describe(obj, indent=0):
    p = '  '*indent
    if isinstance(obj, torch.Tensor):
        print(f'{p}Tensor shape={list(obj.shape)}')
    elif isinstance(obj, dict):
        print(f'{p}dict keys={list(obj.keys())}')
        for k,v in obj.items():
            print(f'{p}  [{k}]:'); _describe(v, indent+2)
    elif isinstance(obj, (list,tuple)):
        print(f'{p}{type(obj).__name__} len={len(obj)}')
        for i,v in enumerate(obj):
            print(f'{p}  [{i}]:'); _describe(v, indent+2)
    else:
        print(f'{p}{type(obj).__name__}')

_describe(_out)

try:
    cl, pl, pr = extract_preds(_out)
    print(f'\nextract_preds OK')
    print(f'  preds:         {list(pr.shape)}')
    print(f'  class_logits:  {list(cl.shape)}')
    print(f'  person_logits: {list(pl.shape)}')
    print('\nAll loss functions ready. Run cells below.')
except Exception as e:
    print(f'\nextract_preds FAILED: {e}')
    print('Paste output above and send to Baek to fix.')

In [ ]:
# ============================================================
# ATTACK 1: FGSM (single step, fastest, weakest)
# One forward pass + one backward pass.
# Good for proving the pipeline works.
# ============================================================

def fgsm_attack(img_rgb, patch_init, box, epsilon):
    img_tensor = img_to_tensor(img_rgb)
    img_shape  = (img_rgb.shape[0], img_rgb.shape[1])
    patch      = patch_init.clone().requires_grad_(True)

    img_patched = apply_patch(img_tensor, patch, box)

    torch_model.train()
    raw_out = get_raw_output(img_patched)
    torch_model.eval()

    loss, components = combined_loss(raw_out, box, img_shape)
    loss.backward()

    with torch.no_grad():
        # Subtract: minimize confidence
        patch_adv = patch - epsilon * patch.grad.sign()
        patch_adv = patch_adv.clamp(0, 1)

    img_adv    = apply_patch(img_tensor, patch_adv.detach(), box)
    img_adv_np = tensor_to_img(img_adv)
    return patch_adv.detach(), img_adv_np, components

print('Running FGSM...')
torch_model.train()
patch_fgsm, img_fgsm, fgsm_components = fgsm_attack(orig_rgb, patch_init, shirt_box, EPSILON)
torch_model.eval()

_, fgsm_conf = run_inference(img_fgsm, 'After FGSM')
print(f'Baseline: {baseline_conf:.4f} -> FGSM: {fgsm_conf:.4f} | drop: {baseline_conf-fgsm_conf:.4f}')

In [ ]:
# ============================================================
# ATTACK 2: PGD (iterative, stronger)
# Many small gradient steps, each clipped to epsilon ball.
# ============================================================

def pgd_attack(img_rgb, patch_init, box, epsilon, alpha, steps, verbose=True):
    img_tensor = img_to_tensor(img_rgb)
    img_shape  = (img_rgb.shape[0], img_rgb.shape[1])
    patch      = patch_init.clone()
    patch_orig = patch_init.clone().detach()
    loss_history = []

    torch_model.train()
    for step in range(steps):
        patch = patch.detach().requires_grad_(True)
        img_patched = apply_patch(img_tensor, patch, box)
        raw_out = get_raw_output(img_patched)
        loss, components = combined_loss(raw_out, box, img_shape)
        loss_history.append(components['total'])
        loss.backward()

        with torch.no_grad():
            patch = patch - alpha * patch.grad.sign()
            delta = (patch - patch_orig).clamp(-epsilon, epsilon)
            patch = (patch_orig + delta).clamp(0, 1)

        if verbose and (step+1) % 10 == 0:
            print(f'  Step [{step+1}/{steps}] loss={components["total"]:+.4f} '
                  f'conf={components["confidence"]:.4f} '
                  f'unc={components["uncertainty"]:.4f}')

    torch_model.eval()
    img_adv    = apply_patch(img_tensor, patch, box)
    return patch.detach(), tensor_to_img(img_adv), loss_history

print('Running PGD...')
patch_pgd, img_pgd, pgd_history = pgd_attack(
    orig_rgb, patch_init, shirt_box, EPSILON, PGD_ALPHA, PGD_STEPS
)

_, pgd_conf = run_inference(img_pgd, 'After PGD')

plt.figure(figsize=(8,4))
plt.plot(pgd_history, color='red')
plt.xlabel('Step'), plt.ylabel('Loss')
plt.title('PGD Loss Curve'), plt.grid(True), plt.show()

print(f'Baseline: {baseline_conf:.4f} -> PGD: {pgd_conf:.4f} | drop: {baseline_conf-pgd_conf:.4f}')

In [ ]:
# ============================================================
# ATTACK 3: EoT-PGD (iterative + random transforms, strongest)
# Averages gradients over EOT_N random real-world transforms
# per step. Makes patch robust to fabric/lighting/angle variation.
# Uses Adam instead of raw sign updates to handle multi-objective
# loss better.
# ============================================================

def eot_pgd_attack(img_rgb, patch_init, box, epsilon, alpha, steps, eot_n, verbose=True):
    img_tensor = img_to_tensor(img_rgb)
    img_shape  = (img_rgb.shape[0], img_rgb.shape[1])
    patch      = patch_init.clone().requires_grad_(True)
    patch_orig = patch_init.clone().detach()
    optimizer  = torch.optim.Adam([patch], lr=LR)

    history = {'total': [], 'uncertainty': [], 'iou': [], 'confidence': [], 'edge': []}
    best_loss  = float('inf')
    best_patch = patch.detach().clone()

    torch_model.train()
    for step in range(steps):
        optimizer.zero_grad()

        step_comps = {'uncertainty':0, 'iou':0, 'confidence':0, 'edge':0, 'total':0}
        total_loss = torch.tensor(0.0, device=DEVICE)

        for _ in range(eot_n):
            img_patched     = apply_patch(img_tensor, patch, box)
            img_transformed = random_eot_transform(img_patched)
            raw_out         = get_raw_output(img_transformed)
            loss, comps     = combined_loss(raw_out, box, img_shape)
            total_loss      = total_loss + loss
            for k in comps:
                step_comps[k] += comps[k]

        avg_loss = total_loss / eot_n
        avg_loss.backward()
        torch.nn.utils.clip_grad_norm_([patch], max_norm=1.0)
        optimizer.step()

        with torch.no_grad():
            delta = patch.data - patch_orig
            delta = delta.clamp(-epsilon, epsilon)
            patch.data = (patch_orig + delta).clamp(0, 1)

        for k in history:
            history[k].append(step_comps[k] / eot_n)

        if avg_loss.item() < best_loss:
            best_loss  = avg_loss.item()
            best_patch = patch.detach().clone()

        if verbose and (step+1) % 10 == 0:
            ec = {k: step_comps[k]/eot_n for k in step_comps}
            print(f'  Step [{step+1}/{steps}] '
                  f'total={ec["total"]:+.4f} '
                  f'unc={ec["uncertainty"]:+.4f} '
                  f'iou={ec["iou"]:+.4f} '
                  f'conf={ec["confidence"]:+.4f} '
                  f'edge={ec["edge"]:+.4f}')

    torch_model.eval()
    img_adv = apply_patch(img_tensor, best_patch, box)
    return best_patch, tensor_to_img(img_adv), history

print(f'Running EoT-PGD ({EOT_STEPS} steps x {EOT_N} transforms = {EOT_STEPS*EOT_N} forward passes)...')
patch_eot, img_eot, eot_history = eot_pgd_attack(
    orig_rgb, patch_init, shirt_box, EPSILON, EOT_ALPHA, EOT_STEPS, EOT_N
)

_, eot_conf = run_inference(img_eot, 'After EoT-PGD')
print(f'Baseline: {baseline_conf:.4f} -> EoT-PGD: {eot_conf:.4f} | drop: {baseline_conf-eot_conf:.4f}')

In [ ]:
# ============================================================
# FINAL COMPARISON: side by side + numeric summary
# ============================================================

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
images = [orig_rgb,      img_fgsm,   img_pgd,  img_eot]
confs  = [baseline_conf, fgsm_conf,  pgd_conf, eot_conf]
labels = ['Baseline',    'FGSM',     'PGD',    'EoT-PGD']
colors = ['green',       'orange',   'red',    'darkred']

for ax, img, conf, label, color in zip(axes, images, confs, labels, colors):
    ax.imshow(img)
    ax.set_title(f'{label}\nConf: {conf:.3f}', color=color, fontsize=13, fontweight='bold')
    ax.axis('off')

plt.suptitle('Attack Comparison: YOLOv8-seg Person Confidence', fontsize=14)
plt.tight_layout()
plt.savefig('/content/comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '='*55)
print(f'{"Method":<12} {"Conf":<10} {"Drop":<10} {"% Reduction"}')
print('-'*55)
for label, conf in zip(labels, confs):
    drop = baseline_conf - conf
    pct  = drop / baseline_conf * 100 if baseline_conf > 0 else 0
    flag = ' <-- BEST' if conf == min(fgsm_conf, pgd_conf, eot_conf) else ''
    print(f'{label:<12} {conf:<10.4f} {drop:<10.4f} {pct:.1f}%{flag}')
print('='*55)

In [ ]:
# ============================================================
# LOSS CURVES: PGD vs EoT-PGD
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(pgd_history, color='red', label='PGD total loss')
axes[0].axhline(y=baseline_conf, color='green', linestyle=':', label=f'Baseline conf ({baseline_conf:.3f})')
axes[0].set_title('PGD Loss Curve')
axes[0].set_xlabel('Step'), axes[0].set_ylabel('Loss')
axes[0].legend(), axes[0].grid(True)

for k, color in zip(['total','uncertainty','iou','confidence','edge'],
                    ['black','blue','red','orange','purple']):
    axes[1].plot(eot_history[k], label=k, color=color,
                 linewidth=2 if k=='total' else 1,
                 linestyle='-' if k=='total' else '--')
axes[1].set_title('EoT-PGD: All Loss Components')
axes[1].set_xlabel('Step'), axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=8), axes[1].grid(True)

plt.suptitle('Loss Curves', fontsize=13)
plt.tight_layout()
plt.savefig('/content/loss_curves.png', dpi=150)
plt.show()
print('Saved: /content/loss_curves.png')

In [ ]:
# ============================================================
# SAVE PATCHES
# ============================================================

def save_patch(patch_tensor, path):
    p = patch_tensor.squeeze(0).permute(1,2,0).cpu().detach().numpy()
    p = (p * 255).clip(0,255).astype(np.uint8)
    Image.fromarray(p).save(path)
    # Also save 300x300 upscale for print preview
    large = cv2.resize(p, (300,300), interpolation=cv2.INTER_NEAREST)
    Image.fromarray(large).save(path.replace('.png', '_300x300.png'))
    print(f'Saved: {path}')

save_patch(patch_fgsm, '/content/patch_fgsm.png')
save_patch(patch_pgd,  '/content/patch_pgd.png')
save_patch(patch_eot,  '/content/patch_eot.png')

# Visualize all 3 patches side by side
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, pt, label in zip(axes,
                          [patch_fgsm, patch_pgd, patch_eot],
                          ['FGSM', 'PGD', 'EoT-PGD']):
    p = pt.squeeze(0).permute(1,2,0).cpu().detach().numpy()
    p = (p * 255).clip(0,255).astype(np.uint8)
    ax.imshow(cv2.resize(p, (150,150), interpolation=cv2.INTER_NEAREST))
    ax.set_title(label)
    ax.axis('off')
plt.suptitle('Learned Adversarial Patches')
plt.tight_layout()
plt.show()
print('\nAll done. Download patches from the Colab file browser on the left.')